# 01 — Data Preparation

In [ ]:
# ── Configuration and paths ──
import pandas as pd
import numpy as np
from pathlib import Path
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

# Project paths
BASE_DIR = Path(".")
OTU_DIR  = BASE_DIR / "otu_tables"
META_DIR = BASE_DIR / "metadata"
OUT_DIR  = BASE_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

# Study registry
STUDIES = [
    "Baxter_2019",
    "Dahl_2016",
    "Deehan_2016",
    "Healey_2018",
    "Hooda_2012",
    "Kovatcheva_2015",
    "Liu_2017",
    "Morales_2016",
    "Rasmussen_2017",
    "Tap_2015",
    "Venkataraman_2016",
]

# Studies with integer sample IDs — need string conversion before join
INTEGER_ID_STUDIES = {"Dahl_2016", "Morales_2016"}

print("Paths and config loaded.")
print(f"OTU dir  : {OTU_DIR}")
print(f"Meta dir : {META_DIR}")
print(f"Output   : {OUT_DIR}")

In [ ]:
# ── Load OTU table from per-study TSV ──
def load_otu_table(study_name):
    filepath = OTU_DIR / f"{study_name}_OTU.tsv"

    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    clean_lines = [l for l in lines if l.startswith('#') or
                   (len(l) > 0 and l[0].isalnum())]

    df = pd.read_csv(StringIO(''.join(clean_lines)), sep='\t', index_col=0)

    # Case-insensitive taxonomy column detection
    tax_col = next((c for c in df.columns if c.lower() == 'taxonomy'), None)
    if tax_col is None:
        raise ValueError(f"No taxonomy column in {study_name}. Columns: {list(df.columns)}")

    taxonomy = df[tax_col].copy()
    df = df.drop(columns=[tax_col])

    if study_name in INTEGER_ID_STUDIES:
        df.columns = df.columns.astype(str)

    return df, taxonomy

In [ ]:
# ── Load and harmonise metadata ──
def load_metadata(study_name):
    filepath = META_DIR / f"{study_name}_metadata.txt"
    meta_raw = pd.read_csv(filepath, sep='\t', dtype=str, encoding='latin-1')

    # All fields extracted by position — header is shifted due to unnamed leading columns
    meta = pd.DataFrame()
    meta['sample_id_2'] = meta_raw.iloc[:, 2]   # real sample ID
    meta['subject_id']  = meta_raw.iloc[:, 3]   # labeled 'study' in header
    meta['treatment']   = meta_raw.iloc[:, 4]   # labeled 'sample_id_2' in header
    meta['timepoint']   = meta_raw.iloc[:, 5]   # labeled 'subject_id' in header
    meta['fiber_type']  = meta_raw.iloc[:, 9]   # labeled 'timepoint_id' in header
    meta['gender']      = meta_raw.iloc[:, 15]
    meta['age']         = meta_raw.iloc[:, 16]
    meta['study']       = study_name

    # Morales: drop Orlistat rows
    if study_name == "Morales_2016":
        before = len(meta)
        meta = meta[~meta['treatment'].isin(['Orlistat', 'Orlistat-fiber'])]
        print(f"  Morales: dropped {before - len(meta)} Orlistat rows, {len(meta)} remaining")

    # Integer ID studies: ensure string type
    if study_name in INTEGER_ID_STUDIES:
        meta['sample_id_2'] = meta['sample_id_2'].astype(str)

    return meta

In [ ]:
# ── Join OTU + metadata per study ──
def join_study(study_name):
    otu, taxonomy = load_otu_table(study_name)
    meta = load_metadata(study_name)

    otu_samples  = set(otu.columns)
    meta_samples = set(meta['sample_id_2'].values)
    common       = otu_samples & meta_samples
    only_otu     = otu_samples - meta_samples
    only_meta    = meta_samples - otu_samples

    if only_otu:
        print(f"  [{study_name}] {len(only_otu)} OTU samples not in metadata — dropped")
    if only_meta:
        print(f"  [{study_name}] {len(only_meta)} metadata rows not in OTU table — dropped")

    otu_common  = otu[sorted(common)]
    meta_common = meta[meta['sample_id_2'].isin(common)].copy()

    return otu_common, meta_common, taxonomy

In [ ]:
# ── Load all 11 studies ──
all_counts   = {}
all_meta     = []
all_taxonomy = {}

for study in STUDIES:
    print(f"Loading {study}...")
    counts, meta, taxonomy = join_study(study)
    all_counts[study] = counts
    all_meta.append(meta)
    all_taxonomy.update(taxonomy.to_dict())
    print(f"  → {counts.shape[1]} samples, {counts.shape[0]} OTUs")

print("\nAll studies loaded.")

In [ ]:
# ── Deduplicate OTU IDs and merge counts across studies ──
for study in all_counts:
    df = all_counts[study]
    n_dupes = df.index.duplicated().sum()
    if n_dupes > 0:
        print(f"{study}: removing {n_dupes} duplicate OTU IDs")
        all_counts[study] = df[~df.index.duplicated(keep='first')]

merged_counts = pd.concat(all_counts.values(), axis=1, join='outer').fillna(0)
print(f"Merged OTU table : {merged_counts.shape[0]} OTUs × {merged_counts.shape[1]} samples")

In [ ]:
# ── Merge metadata and build taxonomy table ──
merged_meta = pd.concat(all_meta, axis=0, ignore_index=True)
print(f"Merged metadata  : {merged_meta.shape[0]} rows × {merged_meta.shape[1]} cols")

taxonomy_df = pd.DataFrame.from_dict(all_taxonomy, orient='index', columns=['taxonomy'])
taxonomy_df.index.name = 'OTU_ID'
print(f"Taxonomy table   : {taxonomy_df.shape[0]} unique OTUs")

In [ ]:
# ── Fix Kovatcheva baseline fiber_type labels ──
kov_mask = (
    (merged_meta['study'] == 'Kovatcheva_2015') &
    (merged_meta['fiber_type'] == 'baseline')
)
print(f"Kovatcheva baseline rows to fix: {kov_mask.sum()}")

kov_after = merged_meta[
    (merged_meta['study'] == 'Kovatcheva_2015') &
    (merged_meta['timepoint'] == 'after')
][['subject_id', 'fiber_type']].drop_duplicates()

kov_lookup = dict(zip(kov_after['subject_id'], kov_after['fiber_type']))

def fix_kovatcheva(row):
    if row['study'] == 'Kovatcheva_2015' and row['fiber_type'] == 'baseline':
        return kov_lookup.get(row['subject_id'], 'baseline')
    return row['fiber_type']

merged_meta['fiber_type'] = merged_meta.apply(fix_kovatcheva, axis=1)

remaining = (
    (merged_meta['study'] == 'Kovatcheva_2015') &
    (merged_meta['fiber_type'] == 'baseline')
).sum()
print(f"Remaining baseline entries after fix: {remaining} (should be 0)")

In [ ]:
# ── Create globally unique sample IDs ──
merged_meta['sample_uid'] = merged_meta['study'] + '_' + merged_meta['sample_id_2']

# Apply same to merged_counts columns
study_prefix_map = {}
for study, df in all_counts.items():
    for col in df.columns:
        study_prefix_map[col] = f"{study}_{col}"

merged_counts.columns = [study_prefix_map.get(col, col) for col in merged_counts.columns]

# Verify uniqueness
dupes_uid = merged_meta['sample_uid'].duplicated().sum()
print(f"Duplicate sample_uid: {dupes_uid} (should be 0)")
print(f"Sample columns match metadata rows: {merged_counts.shape[1] == merged_meta.shape[0]}")

In [ ]:
# ── Validation report ──
report_lines = []

def log(msg):
    print(msg)
    report_lines.append(msg)

log("=" * 60)
log("DATA PREPARATION VALIDATION REPORT")
log("=" * 60)

log("\n── Sample counts per study ──")
for study in STUDIES:
    actual = merged_meta[merged_meta['study'] == study].shape[0]
    log(f"  {study:<22} {actual:>5} samples")

log("\n── Duplicate sample_uid ──")
dupes = merged_meta['sample_uid'].duplicated().sum()
log(f"  Duplicates: {dupes}  {'OK' if dupes == 0 else 'WARNING'}")

log("\n── Timepoint values ──")
tp_vals = merged_meta['timepoint'].unique()
log(f"  Unique values: {sorted(tp_vals)}")
unexpected_tp = set(tp_vals) - {'before', 'after'}
log(f"  Unexpected: {unexpected_tp if unexpected_tp else 'None — OK'}")

log("\n── Treatment values ──")
tr_vals = merged_meta['treatment'].unique()
log(f"  Unique values: {sorted(tr_vals)}")
unexpected_tr = set(tr_vals) - {'fiber', 'control'}
log(f"  Unexpected: {unexpected_tr if unexpected_tr else 'None — OK'}")

log("\n── Kovatcheva fiber_type check ──")
kov_baseline = ((merged_meta['study'] == 'Kovatcheva_2015') & 
                (merged_meta['fiber_type'] == 'baseline')).sum()
log(f"  Remaining baseline rows: {kov_baseline}  {'OK' if kov_baseline == 0 else 'WARNING'}")

log(f"\n── Totals ──")
log(f"  Total samples : {merged_meta.shape[0]}")
log(f"  Total OTUs    : {merged_counts.shape[0]}")
log(f"  OTU matrix    : {merged_counts.shape[0]} × {merged_counts.shape[1]}")
log("=" * 60)

In [ ]:
# ── Save outputs ──
merged_counts.to_csv(OUT_DIR / "merged_otu_counts.csv")
print("Saved: merged_otu_counts.csv")

merged_meta.to_csv(OUT_DIR / "merged_metadata.csv", index=False)
print("Saved: merged_metadata.csv")

taxonomy_df.to_csv(OUT_DIR / "taxonomy_table.csv")
print("Saved: taxonomy_table.csv")

with open(OUT_DIR / "data_prep_report.txt", 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))
print("Saved: data_prep_report.txt")

print("\nStage 1 complete.")